# Deep Belief Network for Customer Behavior Representation

This notebook demonstrates an educational implementation of a **Deep Belief Network (DBN)** constructed by stacking two **Bernoulli Restricted Boltzmann Machines (RBMs)**.

### Workflow Overview
1. **Customer Behavior Data**: 6 numerical behavioral metrics
2. **Preprocessing**: Median-based thresholding into binary inputs
3. **RBM Layer 1**: $6 \rightarrow 4$ hidden units
4. **RBM Layer 2**: $4 \rightarrow 2$ hidden units
5. **Learned 2D Representation**: Low-dimensional feature extraction
6. **K-Means Clustering**: Exploratory grouping ($k=3$)
7. **Cluster Analysis & Visualization**: Pattern discovery and behavioral insights

## 1. Import Libraries
Import core libraries for data processing, neural modeling, clustering, and visualization.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.neural_network import BernoulliRBM
from sklearn.cluster import KMeans

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Dataset
Load the unlabeled customer behavior dataset containing 100 customer records.

In [2]:
data_path = Path("../dataset/customer_data.csv")
if not data_path.exists():
    data_path = Path("dataset/customer_data.csv")

df = pd.read_csv(data_path)
print(f"Loaded dataset with {df.shape[0]} samples and {df.shape[1]} features.")

Loaded dataset with 100 samples and 6 features.


## 3. Display Dataset
Inspect the first few records and statistical summary of customer behavior features.

In [3]:
print("First 5 customer records:")
print(df.head())

print("\nSummary statistics:")
print(df.describe().round(2))

First 5 customer records:
   website_visits  purchase_frequency  ...  support_requests  discount_usage
0               9                   1  ...                 7               2
1              12                   5  ...                 1              10
2              10                   2  ...                 4               4
3              18                   3  ...                 1               9
4              19                   3  ...                 1               7

[5 rows x 6 columns]

Summary statistics:
       website_visits  purchase_frequency  ...  support_requests  discount_usage
count          100.00              100.00  ...            100.00          100.00
mean            17.49                6.03  ...              2.40            4.70
std              8.79                5.31  ...              2.28            3.27
min              3.00                0.00  ...              0.00            0.00
25%             10.00                2.00  ...              1.00

## 4. Preprocess Features
Bernoulli RBMs model binary stochastic visible units. We convert the continuous/discrete customer behavior features into binary values using a simple, explainable **median threshold**:

$$\text{binary\_value} = \begin{cases} 1 & \text{if } x \ge \text{median}(x) \\ 0 & \text{otherwise} \end{cases}$$

In [4]:
feature_cols = [
    "website_visits",
    "purchase_frequency",
    "average_spend",
    "app_usage",
    "support_requests",
    "discount_usage"
]

features = df[feature_cols].copy()
medians = features.median()
X_bin = (features >= medians).astype(int).values

print("Feature Medians:")
for col, med in medians.items():
    print(f"  {col}: {med:.2f}")

print(f"\nBinary preprocessed data shape: {X_bin.shape}")
print("Sample preprocessed binary rows (first 3):")
print(X_bin[:3])

Feature Medians:
  website_visits: 16.00
  purchase_frequency: 3.50
  average_spend: 49.59
  app_usage: 19.60
  support_requests: 2.00
  discount_usage: 4.00

Binary preprocessed data shape: (100, 6)
Sample preprocessed binary rows (first 3):
[[0 0 1 0 1 0]
 [0 1 0 1 0 1]
 [0 0 1 0 1 1]]


## 5. Train RBM 1
Train the first Restricted Boltzmann Machine with **6 visible inputs** and **4 hidden units** using Contrastive Divergence.

In [5]:
rbm1 = BernoulliRBM(
    n_components=4,
    learning_rate=0.1,
    n_iter=50,
    batch_size=10,
    random_state=RANDOM_STATE,
    verbose=False
)
rbm1.fit(X_bin)
print("RBM 1 Training completed.")
print("RBM 1 weight matrix shape (components x features):", rbm1.components_.shape)

RBM 1 Training completed.
RBM 1 weight matrix shape (components x features): (4, 6)


## 6. Generate First Hidden Representation
Transform the binary customer input through RBM 1 to obtain 4-dimensional hidden unit activations.

In [6]:
hidden_1 = rbm1.transform(X_bin)
print(f"Layer 1 hidden representation shape: {hidden_1.shape}")
print("Sample Layer 1 activations (first 3 records):")
print(np.round(hidden_1[:3], 4))

Layer 1 hidden representation shape: (100, 4)
Sample Layer 1 activations (first 3 records):
[[0.383  0.3672 0.6258 0.5697]
 [0.6429 0.6618 0.2279 0.2495]
 [0.3778 0.3628 0.6448 0.5699]]


## 7. Train RBM 2
Stack the second RBM on top of Layer 1. It takes the **4 hidden units** of Layer 1 as visible inputs and maps them down to **2 hidden units**.

In [7]:
rbm2 = BernoulliRBM(
    n_components=2,
    learning_rate=0.1,
    n_iter=50,
    batch_size=10,
    random_state=RANDOM_STATE,
    verbose=False
)
rbm2.fit(hidden_1)
print("RBM 2 Training completed.")
print("RBM 2 weight matrix shape:", rbm2.components_.shape)

RBM 2 Training completed.
RBM 2 weight matrix shape: (2, 4)


## 8. Generate Final Hidden Representation
Transform Layer 1 activations through RBM 2 to obtain the final 2-dimensional hidden representation.

In [8]:
hidden_2 = rbm2.transform(hidden_1)
print(f"Final learned hidden representation shape: {hidden_2.shape}")
h_df = pd.DataFrame(hidden_2, columns=["hidden_dim_1", "hidden_dim_2"])
print("\nFirst 5 customer hidden representations:")
print(h_df.head())

Final learned hidden representation shape: (100, 2)

First 5 customer hidden representations:
   hidden_dim_1  hidden_dim_2
0      0.468364      0.468361
1      0.497791      0.497424
2      0.467566      0.467574
3      0.485247      0.485033
4      0.497061      0.496701


## 9. Analyze Hidden Representation
Examine the statistical properties of the learned 2-dimensional feature space.

In [9]:
print("Hidden Representation Statistics:")
print(h_df.describe().round(4))

plt.figure(figsize=(7, 5))
plt.scatter(h_df["hidden_dim_1"], h_df["hidden_dim_2"], color="#2b5c8f", alpha=0.75, s=45)
plt.title("Learned Customer Representation (DBN Hidden Space)", fontsize=12)
plt.xlabel("Hidden Dimension 1")
plt.ylabel("Hidden Dimension 2")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

Hidden Representation Statistics:
       hidden_dim_1  hidden_dim_2
count      100.0000      100.0000
mean         0.4869        0.4866
std          0.0189        0.0186
min          0.4595        0.4596
25%          0.4684        0.4684
50%          0.4853        0.4851
75%          0.5058        0.5054
max          0.5104        0.5099


## 10. Apply K-Means
Apply K-Means clustering ($k=3$) strictly for exploratory pattern analysis of the learned 2D representation.

> **Note**: These clusters are discovered behavioral patterns, not ground-truth labels.

In [10]:
kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(hidden_2)
clustered_df = df.copy()
clustered_df["cluster"] = cluster_labels

print("Cluster Counts:")
counts = pd.Series(cluster_labels).value_counts().sort_index()
for c, count in counts.items():
    pct = (count / len(cluster_labels)) * 100
    print(f"  Cluster {c}: {count} customers ({pct:.1f}%)")

Cluster Counts:
  Cluster 0: 35 customers (35.0%)
  Cluster 1: 41 customers (41.0%)
  Cluster 2: 24 customers (24.0%)


## 11. Display Cluster Statistics
Calculate and interpret the mean values of the original customer features for each discovered cluster.

In [11]:
cluster_means = clustered_df.groupby("cluster")[feature_cols].mean().round(2)
print("Mean Feature Values by Cluster:")
print(cluster_means)

print("\n--- Behavioral Interpretations ---")
for c in range(3):
    means = cluster_means.loc[c]
    print(f"\nCluster {c}:")
    print(f"  Visits: {means['website_visits']}, Purchases: {means['purchase_frequency']}")
    print(f"  Avg Spend: ${means['average_spend']}, App Usage: {means['app_usage']}")
    print(f"  Support Requests: {means['support_requests']}, Discount Usage: {means['discount_usage']}")

Mean Feature Values by Cluster:
         website_visits  purchase_frequency  ...  support_requests  discount_usage
cluster                                      ...                                  
0                 27.43               12.37  ...              1.00            2.91
1                 10.17                2.12  ...              4.27            4.12
2                 15.50                3.46  ...              1.25            8.29

[3 rows x 6 columns]

--- Behavioral Interpretations ---

Cluster 0:
  Visits: 27.43, Purchases: 12.37
  Avg Spend: $121.27, App Usage: 45.29
  Support Requests: 1.0, Discount Usage: 2.91

Cluster 1:
  Visits: 10.17, Purchases: 2.12
  Avg Spend: $45.19, App Usage: 11.05
  Support Requests: 4.27, Discount Usage: 4.12

Cluster 2:
  Visits: 15.5, Purchases: 3.46
  Avg Spend: $37.85, App Usage: 19.08
  Support Requests: 1.25, Discount Usage: 8.29


## 12. Visualize Clusters
Visualize the clusters in the 2D DBN hidden space along with cluster centroids.

In [12]:
plt.figure(figsize=(7, 5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for c in np.unique(cluster_labels):
    mask = cluster_labels == c
    plt.scatter(
        hidden_2[mask, 0],
        hidden_2[mask, 1],
        label=f"Cluster {c}",
        color=colors[c % len(colors)],
        alpha=0.8,
        s=50
    )

centers = kmeans.cluster_centers_
plt.scatter(
    centers[:, 0],
    centers[:, 1],
    marker="X",
    s=130,
    color="red",
    edgecolors="black",
    linewidths=1.2,
    label="Centroids"
)

plt.title("Customer Behavior Clusters (DBN Hidden Space)", fontsize=12)
plt.xlabel("Hidden Dimension 1")
plt.ylabel("Hidden Dimension 2")
plt.legend(frameon=True)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 13. Conclusion

1. **Unsupervised Representation Learning**: The stacked Bernoulli RBMs successfully compressed 6 customer behavior features into an informative 2D hidden space without using any labels.
2. **Discovered Patterns**: K-Means clustering in the learned latent space revealed 3 distinct customer behavior profiles:
   - **Cluster 0**: High digital engagement, frequent shoppers, and highest average spend.
   - **Cluster 1**: Lower engagement with higher support inquiry frequency.
   - **Cluster 2**: Moderate engagement with high discount and coupon reliance.
3. **Academic Relevance**: The simple $6 \rightarrow 4 \rightarrow 2$ architecture demonstrates clear layer-wise greedy pre-training principles suitable for educational demonstration.